# 保险保单受益人变更检索系统 — 完整实现教程

## RAG-based Insurance Policy Beneficiary Change Retrieval System

本教程将带你从零构建一个完整的 **RAG（检索增强生成）** 系统，用于保险保单受益人变更检索。

### 业务场景
保险公司持有海量多页 TIF 格式的历史保单，当发生受益人变更时，理赔人员需要快速定位：
1. **哪份保单**发生了变更？
2. **变更发生在哪一页、哪一段**？
3. **变更前后的具体内容是什么**？

### 系统架构（4 层）
```
① 解析层 (ETL):  OCR → 元数据提取 → 父子分块
② 索引层:         向量索引 (Dense) + 全文索引 (BM25)
③ 检索层:         RRF 融合 + Metadata 过滤 + Reranker 精排
④ 生成层:         Prompt 构造 + LLM 生成
```

### 技术栈
| 模块 | 技术选型 |
|------|----------|
| 向量模型 | all-MiniLM-L6-v2 (384-dim) |
| 全文检索 | jieba + rank_bm25 |
| 精排模型 | BAAI/bge-reranker-v2-m3 |
| LLM | DeepSeek (deepseek-v4-flash) |
| API 框架 | FastAPI + uvicorn |
| 数据库 | PostgreSQL + pgvector（可选） |
| 前端 | Streamlit |
| 部署 | Docker Compose |


## 1. 环境准备与依赖安装

首先安装所有必需的 Python 包。本系统核心依赖分为以下几类：

- **LLM/API**: openai, fastapi, uvicorn, pydantic
- **向量/ML**: sentence-transformers, torch, transformers
- **中文分词/检索**: jieba, rank-bm25
- **数据库**: psycopg2-binary（PostgreSQL）
- **前端**: streamlit
- **工具**: numpy, python-dotenv, python-multipart


In [ ]:
# 安装依赖（如已在环境中可跳过）
# pip install fastapi uvicorn openai sentence-transformers jieba rank-bm25
# pip install numpy pydantic python-multipart psycopg2-binary streamlit python-dotenv
# pip install torch transformers  # 用于 Reranker

# 导入所有必需库
import re
import json
import os
import numpy as np
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Tuple, Any

print("✅ 所有库导入成功")


## 2. 数据模型定义

首先定义核心数据结构。这些模型贯穿整个系统：

- **`PolicyMeta`**: 保单的元数据（保单号、保险公司、投保人、受益人列表、批单历史）
- **`Chunk`**: 分块后的保单文本片段（子块用于检索，父块用于上下文）


In [ ]:
@dataclass
class PolicyMeta:
    """保单元数据：从保单文本中提取的结构化信息"""
    id: str                              # 保单号
    insurer: str                         # 保险公司
    applicant: str = ""                  # 投保人
    insured: str = ""                    # 被保险人
    product: str = ""                    # 产品名称
    premium: str = ""                    # 保费
    eff_date: str = ""                   # 生效日期
    beneficiaries: List[Dict] = field(default_factory=list)   # 当前受益人列表
    endorsements: List[Dict] = field(default_factory=list)    # 历史批单

    def to_prompt(self) -> str:
        """格式化为 LLM 可读的文本"""
        lines = [f"保单号：{self.id}", f"保险公司：{self.insurer}",
                 f"投保人：{self.applicant}", f"被保险人：{self.insured}",
                 f"产品名称：{self.product}", f"保费：{self.premium}",
                 f"生效日期：{self.eff_date}"]
        if self.beneficiaries:
            lines.append("当前受益人：")
            for b in self.beneficiaries:
                lines.append(f"  - {b.get('name','?')} 比例 {b.get('ratio','?')} "
                             f"与被保人关系 {b.get('relation','?')}")
        if self.endorsements:
            lines.append("历史批单：")
            for e in self.endorsements:
                lines.append(f"  - 批单号 {e.get('id','?')} ({e.get('date','?')})")
                lines.append(f"    变更内容：{e.get('change','?')}")
        return "\n".join(lines)


@dataclass
class Chunk:
    """保单文本片段"""
    cid: str          # chunk id，格式: "c-保单号-页码-序号"
    pid: str          # policy_id，所属保单
    page: int         # 页码
    type: str         # "parent"（父块）或 "child"（子块）
    parent_id: str = ""   # 父块 ID（仅子块使用）
    heading: str = ""     # 段落标题
    text: str = ""        # 文本内容
    embedding: Optional[List[float]] = None  # 向量（可选）

print("✅ 数据模型定义完成")
print(f"   PolicyMeta: {len(field(PolicyMeta).__dataclass_fields__)} 个字段")
print(f"   Chunk: {len(field(Chunk).__dataclass_fields__)} 个字段")


## 3. 构造模拟保单数据

PoC 阶段使用 3 份模拟保单（Markdown 格式），覆盖三种典型场景：

| 保单 | 场景 | 保单号 | 保险公司 |
|------|------|--------|----------|
| A | 标准受益人变更 | P0242025-1883 | 中国平安人寿保险 |
| B | 多次受益人变更 | TPK-2023-004517 | 中国太平洋人寿保险 |
| C | 无受益人变更 | PCI-2024-7721 | 中国人寿保险 |

每份保单由多页构成，每页包含 Markdown 格式的文本和模拟的 OCR 置信度。


In [ ]:
import numpy as np
np.random.seed(42)

def _page(pn: int, text: str) -> dict:
    """创建一个模拟保单页面"""
    return {
        "page": pn,
        "image": f"mock_tif/policy_{pn:04d}.tif",
        "md": text,
        "ocr_conf": round(0.92 + np.random.random() * 0.07, 3)
    }

# ── 保单 A：标准受益人变更（张美玲 60%）──
POLICY_A = {
    "id": "P0242025-1883",
    "insurer": "中国平安人寿保险",
    "pages": [
        _page(1, (
            "中国平安人寿保险股份有限公司\n"
            "┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈\n"
            "保险单（副本）\n\n"
            "**保单号**：P0242025-1883\n"
            "**投保人**：王建国\n"
            "**被保险人**：王建国\n"
            "**产品名称**：平安福终身寿险（2024版）\n"
            "**基本保额**：¥500,000.00\n"
            "**保费**：¥8,230.00 / 年\n"
            "**生效日期**：2024-01-15\n\n"
            "受益人信息：\n"
            "  1. 李芳（配偶）100% 受益比例\n"
            "  2. 王浩（长子）—— 顺位：第二顺位"
        )),
        _page(2, (
            "**保险条款（节选）**\n\n"
            "第 2.1 条 保险责任\n"
            "在本合同有效期内，本公司承担以下保险责任：\n\n"
            "一、身故保险金\n"
            "被保险人身故，本公司按基本保险金额给付身故保险金，本合同终止。\n\n"
            "第 3.1 条 受益人指定和变更\n"
            "受益人由被保险人或投保人指定。投保人指定和变更受益人时须经被保险人书面同意。"
        )),
        _page(3, (
            "**批单 BG2024-00137**\n"
            "批单号：BG2024-00137\n"
            "申请日期：2024-08-20\n"
            "生效日期：2024-08-21\n\n"
            "**变更事项**：受益人变更\n\n"
            "**变更前**：\n"
            "  第一顺位：李芳（配偶）受益比例 100%\n\n"
            "**变更后**：\n"
            "  第一顺位：张美玲（配偶）受益比例 60%\n"
            "  第一顺位：李芳（前配偶）受益比例 40%\n\n"
            "**变更原因**：婚姻关系变更及家庭规划调整\n"
            "**核保意见**：同意"
        )),
        _page(4, (
            "**批单 BG2024-00137（续）**\n\n"
            "受益人信息汇总（变更后）：\n"
            "  保单号：P0242025-1883\n"
            "  投保人：王建国\n\n"
            "  受益人明细：\n"
            "  ┌──────────────┬────────┬────────┬──────────┐\n"
            "  │ 受益人姓名   │ 与被保人关系 │ 受益比例 │ 受益顺序 │\n"
            "  ├──────────────┼────────┼────────┼──────────┤\n"
            "  │ 张美玲       │ 配偶   │ 60%    │ 第一顺位 │\n"
            "  │ 李芳         │ 前配偶 │ 40%    │ 第一顺位 │\n"
            "  └──────────────┴────────┴────────┴──────────┘\n\n"
            "保险公司签章：中国平安人寿保险股份有限公司\n"
            "日期：2024-08-21"
        )),
    ],
}

# ── 保单 B：多次受益人变更 ──
POLICY_B = {
    "id": "TPK-2023-004517",
    "insurer": "中国太平洋人寿保险",
    "pages": [
        _page(1, ("中国太平洋人寿保险股份有限公司\n┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈\n"
                  "保险单\n\n**保单号**：TPK-2023-004517\n**投保人**：张伟\n"
                  "**被保险人**：张伟\n**产品名称**：太平洋金佑人生终身寿险（分红型）\n"
                  "**基本保额**：¥300,000.00\n**保费**：¥4,560.00 / 年\n"
                  "**生效日期**：2023-06-01\n\n受益人信息：\n  1. 陈静（配偶）100% 受益比例")),
        _page(2, ("**批单 BG2024-00892**\n批单号：BG2024-00892\n申请日期：2024-03-15\n"
                  "生效日期：2024-03-16\n\n**变更事项**：受益人变更\n\n**变更前**：\n"
                  "  陈静（配偶）受益比例 100%\n\n**变更后**：\n"
                  "  陈静（配偶）受益比例 60%\n  张明轩（长子）受益比例 40%\n\n"
                  "**变更原因**：家庭新增成员，调整受益分配")),
        _page(3, ("**批单 BG2025-00103**\n批单号：BG2025-00103\n申请日期：2025-02-01\n"
                  "生效日期：2025-02-01\n\n**变更事项**：受益人变更\n\n**变更前**：\n"
                  "  陈静（配偶）受益比例 60%\n  张明轩（长子）受益比例 40%\n\n"
                  "**变更后**：\n  陈静（配偶）受益比例 50%\n"
                  "  张明轩（长子）受益比例 30%\n  张明悦（长女）受益比例 20%\n\n"
                  "**变更原因**：新生儿出生，重新分配受益比例\n**核保意见**：同意")),
    ],
}

# ── 保单 C：无受益人变更 ──
POLICY_C = {
    "id": "PCI-2024-7721",
    "insurer": "中国人寿保险",
    "pages": [
        _page(1, ("中国人寿保险股份有限公司\n┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈┈\n"
                  "国寿福终身寿险保险单\n\n**保单号**：PCI-2024-7721\n**投保人**：刘强\n"
                  "**被保险人**：刘强\n**产品名称**：国寿福终身寿险（至尊版）\n"
                  "**基本保额**：¥1,000,000.00\n**保费**：¥12,800.00 / 年\n"
                  "**生效日期**：2024-09-01\n\n受益人信息：\n"
                  "  1. 赵丽华（配偶）受益比例 100%")),
        _page(2, ("**保险条款（节选）**\n\n第 1 条 保险责任\n"
                  "被保险人身故，本公司按基本保险金额给付身故保险金。\n\n"
                  "第 6 条 受益人\n受益人的指定和变更须经被保险人书面同意。")),
    ],
}

POLICIES = [POLICY_A, POLICY_B, POLICY_C]
print(f"✅ 模拟数据就绪：{len(POLICIES)} 份保单")
for p in POLICIES:
    print(f"   {p['id']} - {p['insurer']} ({len(p['pages'])} 页)")


## 4. 解析层 — 元数据提取 (Metadata Extraction)

**目标**：从保单文本中提取结构化字段。

使用正则表达式从 Markdown 文本中提取：
- **保单基本信息**：保单号、保险公司、投保人、被保险人、产品名称、保费、生效日期
- **受益人信息**：姓名、关系、比例、顺序
- **批单历史**：批单号、日期、变更前/后内容、变更原因


In [ ]:
# 正则模式定义
PATTERNS = {
    "policy_id": re.compile(r"\*\*保单号\*\*[：:]?\s*(\S+)"),
    "insurer": re.compile(r"^(中国[^（\n]+)(?:[（(]|\s*保险)"),
    "applicant": re.compile(r"\*\*投保人\*\*[：:]?\s*(\S+)"),
    "insured": re.compile(r"\*\*被保险人\*\*[：:]?\s*(\S+)"),
    "product": re.compile(r"\*\*产品名称\*\*[：:]?\s*(.+)"),
    "premium": re.compile(r"\*\*保费\*\*[：:]?\s*(\S+)"),
    "eff_date": re.compile(r"\*\*生效日期\*\*[：:]?\s*(\S+)"),
    "beneficiary_line": re.compile(
        r"(\d+\s*[.、．]?\s*)(\S+)\s*[（(]?(.+?)[）)]?\s*(\d+%)?\s*受益", re.MULTILINE),
    "endorsement_id": re.compile(r"批单号[：:]?\s*(\S+)"),
    "change_before": re.compile(r"变更前[：:]\s*(.+?)(?=\n\n|\n\*\*)", re.DOTALL),
    "change_after": re.compile(r"变更后[：:]\s*(.+?)(?=\n\n|\n\*\*|\Z)", re.DOTALL),
    "change_reason": re.compile(r"变更原因[：:]?\s*(.+)"),
}


def extract_metadata(policy: dict) -> PolicyMeta:
    """从保单数据中提取结构化元数据"""
    text = "\n".join(p["md"] for p in policy["pages"])

    def get(key):
        m = PATTERNS[key].search(text)
        return m.group(1).strip() if m else ""

    pid = get("policy_id")
    insurer = PATTERNS["insurer"].search(text)
    insurer = insurer.group(1).strip() if insurer else policy.get("insurer", "")

    meta = PolicyMeta(id=pid, insurer=insurer, applicant=get("applicant"),
                      insured=get("insured"), product=get("product"),
                      premium=get("premium"), eff_date=get("eff_date"))

    # 提取受益人信息
    ben_lines = PATTERNS["beneficiary_line"].findall(text)
    for b in ben_lines:
        meta.beneficiaries.append({
            "name": b[1].strip(),
            "ratio": b[3].strip() if b[3] else "100%",
            "relation": (b[2] + " " + b[3] if b[3] else b[2]).strip().rstrip("）)"),
        })

    # 提取批单历史
    for pg in policy["pages"]:
        t = pg["md"]
        if "批单" not in t:
            continue
        eid = PATTERNS["endorsement_id"].search(t)
        before = PATTERNS["change_before"].search(t)
        after = PATTERNS["change_after"].search(t)
        reason = PATTERNS["change_reason"].search(t)
        if eid:
            entry = {"id": eid.group(1), "date": "", "change": "",
                     "before": "", "after": ""}
            if before:
                entry["before"] = before.group(1).strip().replace("\n", " | ")
            if after:
                entry["after"] = after.group(1).strip().replace("\n", " | ")
            if reason:
                entry["change"] = reason.group(1).strip()
            meta.endorsements.append(entry)

    return meta


# 测试：提取保单 A 的元数据
meta_a = extract_metadata(POLICY_A)
print("=" * 50)
print("📋 保单 A 元数据提取结果")
print("=" * 50)
print(f"保单号：{meta_a.id}")
print(f"保险公司：{meta_a.insurer}")
print(f"投保人：{meta_a.applicant}")
print(f"产品：{meta_a.product}")
print(f"保费：{meta_a.premium}")
print(f"\n当前受益人：")
for b in meta_a.beneficiaries:
    print(f"  - {b['name']} ({b['relation']}) {b['ratio']}")
print(f"\n批单历史：")
for e in meta_a.endorsements:
    print(f"  - {e['id']}")
    print(f"    变更前：{e['before'][:60]}...")
    print(f"    变更后：{e['after'][:60]}...")


## 5. 解析层 — 父子分块 (Parent-Child Chunking)

**为什么需要分块？**
- 保单页面较长，直接检索整页精度低
- **子块 (Child)**：小片段（~300 字符），用于嵌入和检索
- **父块 (Parent)**：更大的上下文（~1500 字符），作为 LLM 输入

**分块策略**：
1. 按 Markdown 标题（`**...**`）分割页面为段落
2. 每个段落生成父块（大块）
3. 每个段落滑动窗口生成子块（小块，有重叠）


In [ ]:
# 分块参数
PARENT_MAX_CHARS = 1500   # 父块最大字符数
CHILD_MAX_CHARS = 300     # 子块最大字符数
CHILD_OVERLAP = 30        # 子块重叠字符数


def _split_into_sections(text: str) -> List[Tuple[str, str]]:
    """按 Markdown 标题分割页面"""
    sections = []
    lines = text.split("\n")
    current_heading, current_body = "", []
    for line in lines:
        if line.startswith("**") and line.endswith("**"):
            if current_body:
                sections.append((current_heading, "\n".join(current_body).strip()))
            current_heading = line.strip("*")
            current_body = []
        else:
            current_body.append(line)
    if current_body:
        sections.append((current_heading, "\n".join(current_body).strip()))
    return sections


def chunk_policy(policy: dict) -> List[Chunk]:
    """对保单执行父子分块"""
    parents, children = [], []
    pid = policy["id"]

    for page in policy["pages"]:
        pn = page["page"]
        sections = _split_into_sections(page["md"])
        for heading, body in sections:
            combined = f"{heading}\n{body}" if heading else body
            if not combined.strip():
                continue
            # 生成父块
            for start in range(0, len(combined), PARENT_MAX_CHARS):
                parent_text = combined[start:start + PARENT_MAX_CHARS]
                pc = Chunk(cid=f"p-{pid}-p{pn}-{len(parents)}", pid=pid,
                           page=pn, type="parent", heading=heading, text=parent_text)
                parents.append(pc)
            # 生成子块（滑动窗口）
            cs_text = body if heading else combined
            cs_text = cs_text.strip()
            for start in range(0, len(cs_text), CHILD_MAX_CHARS - CHILD_OVERLAP):
                chunk_text = cs_text[start:start + CHILD_MAX_CHARS]
                if len(chunk_text) < 20 and start > 0:
                    continue
                cc = Chunk(cid=f"c-{pid}-p{pn}-{len(children)}", pid=pid,
                           page=pn, type="child", heading=heading,
                           parent_id=parents[-1].cid if parents else "",
                           text=chunk_text.strip())
                children.append(cc)
    return parents + children


# 测试分块
chunks_a = chunk_policy(POLICY_A)
children_a = [c for c in chunks_a if c.type == "child"]
parents_a = [c for c in chunks_a if c.type == "parent"]

print(f"✅ 保单 A 分块完成")
print(f"   父块: {len(parents_a)} 个")
print(f"   子块: {len(children_a)} 个")
print(f"\n📄 子块示例（第 3 页，批单变更内容）：")
for c in children_a:
    if c.page == 3 and c.type == "child":
        print(f"   [{c.cid}] {c.text[:150]}...")
        break


## 6. 索引层 — 向量编码 (Embedding)

**原理**：使用 Sentence-BERT 将文本编码为稠密向量。

- 模型：`all-MiniLM-L6-v2`（384 维，~80MB）
- 相似度：余弦相似度（cosine similarity）
- 所有子块在启动时预编码，存入 numpy 矩阵


In [ ]:
from sentence_transformers import SentenceTransformer

# 加载模型（首次会从 HuggingFace 下载约 80MB）
print("🔄 加载 Embedding 模型 all-MiniLM-L6-v2...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✅ 模型加载完成，向量维度: {embedder.get_sentence_embedding_dimension()}")

# ========== 构建完整索引 ==========

# 处理所有保单
all_chunks = []
all_metas = {}
for pol in POLICIES:
    pid = pol["id"]
    all_metas[pid] = extract_metadata(pol)
    all_chunks.extend(chunk_policy(pol))

children = [c for c in all_chunks if c.type == "child"]
print(f"\n📊 总览：{len(all_metas)} 份保单, {len(all_chunks)} 个 chunks, {len(children)} 个子块")

# 编码所有子块
child_texts = [c.text for c in children]
print(f"📐 编码 {len(child_texts)} 个子块...")
dense_matrix = embedder.encode(child_texts, show_progress_bar=True)
print(f"✅ 向量矩阵 shape: {dense_matrix.shape}")


## 7. 索引层 — BM25 全文检索

**原理**：基于词频的经典检索算法，精确匹配关键词。

**为什么需要 BM25？**
- 向量检索擅长语义匹配（"受益比例变更"）
- BM25 擅长精确关键词匹配（"张美玲 60%"）
- 两者互补，混合检索效果更好

**中文分词**：使用 jieba 对文本分词，然后构建 BM25 索引。


In [ ]:
import jieba
from rank_bm25 import BM25Okapi


def tokenize(text: str) -> List[str]:
    """jieba 中文分词，去除停用词"""
    tokens = []
    for w in jieba.cut(text):
        w = w.strip()
        if w and w not in ("", " ", "\n", "|", "---", "**:"):
            tokens.append(w.lower())
    return tokens


# 构建 BM25 索引
tokenized_chunks = [tokenize(c.text) for c in children]
bm25 = BM25Okapi(tokenized_chunks)
vocab_size = len(set(w for t in tokenized_chunks for w in t))
print(f"✅ BM25 索引构建完成")
print(f"   词表大小: ~{vocab_size}")
print(f"   索引文档数: {len(tokenized_chunks)}")


## 8. 检索层 — 混合检索 (RRF 融合)

**核心思想**：结合向量检索和 BM25 的结果，通过 **RRF (Reciprocal Rank Fusion)** 算法融合排序。

$$Score(chunk) = \alpha \cdot \frac{1}{rank_{dense} + K} + (1-\alpha) \cdot \frac{1}{rank_{bm25} + K}$$

- $\alpha$: 向量检索权重（默认 0.5）
- $K$: RRF 常数（默认 60），防止极低排序影响
- 可选参数 `policy_id`: 按保单号精确过滤


In [ ]:
# 检索参数
RRF_K = 60
RRF_ALPHA = 0.5
DENSE_TOP_K = 20
BM25_TOP_K = 20
FINAL_TOP_K = 5


def _dense_search(query: str, top_k: int = DENSE_TOP_K,
                  policy_id: Optional[str] = None) -> List[Tuple[Chunk, float]]:
    """向量检索：余弦相似度"""
    qv = embedder.encode([query])[0]
    scores = np.dot(dense_matrix, qv) / (
        np.linalg.norm(dense_matrix, axis=1) * np.linalg.norm(qv) + 1e-10
    )
    if policy_id:
        mask = np.array([c.pid == policy_id for c in children])
        scores[~mask] = -1
    top = np.argsort(scores)[-top_k:][::-1]
    return [(children[i], float(scores[i])) for i in top if scores[i] > 0]


def _bm25_search(query: str, top_k: int = BM25_TOP_K,
                 policy_id: Optional[str] = None) -> List[Tuple[Chunk, float]]:
    """全文检索：BM25"""
    qt = tokenize(query)
    scores = bm25.get_scores(qt)
    if policy_id:
        for i, c in enumerate(children):
            if c.pid != policy_id:
                scores[i] = -1
    top = np.argsort(scores)[-top_k:][::-1]
    return [(children[i], float(scores[i])) for i in top if scores[i] > 0]


def hybrid_search(query: str, policy_id: Optional[str] = None,
                  top_k: int = FINAL_TOP_K) -> List[Tuple[Chunk, float, str]]:
    """RRF 融合混合检索"""
    dense_res = _dense_search(query, DENSE_TOP_K, policy_id)
    bm25_res = _bm25_search(query, BM25_TOP_K, policy_id)

    # RRF 融合
    scores = defaultdict(float)
    for rank, (ch, _) in enumerate(dense_res):
        scores[ch.cid] += RRF_ALPHA / (rank + RRF_K)
    for rank, (ch, _) in enumerate(bm25_res):
        scores[ch.cid] += (1 - RRF_ALPHA) / (rank + RRF_K)

    ranked = sorted(scores.items(), key=lambda x: -x[1])[:top_k]
    return [(next(c for c in all_chunks if c.cid == cid), sc, "hybrid")
            for cid, sc in ranked]


# ── 测试检索 ──
print("🔍 测试检索：'张美玲受益比例'")
results = hybrid_search("张美玲受益比例", policy_id="P0242025-1883")
print(f"\n找到 {len(results)} 个结果：")
for i, (ch, score, src) in enumerate(results, 1):
    print(f"\n#{i} [得分={score:.4f}] 保单{ch.pid} 第{ch.page}页")
    print(f"   {ch.text[:120]}...")


## 9. 检索层 — Reranker 精排

**为什么需要精排？**
- 向量检索（bi-encoder）速度快但精度有限
- Cross-encoder 对 query+chunk 逐对打分，精度更高
- 作为 Top-N 结果的"二审"环节

**流程**：RRF Top-20 → Cross-encoder 重打分 → Top-5 最终结果


In [ ]:
class Reranker:
    """Cross-encoder 精排模型"""
    def __init__(self, model_name: str = "BAAI/bge-reranker-v2-m3"):
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self._ready = False

    def load(self) -> bool:
        """加载模型（首次需下载 2.27GB）"""
        try:
            from transformers import AutoModelForSequenceClassification, AutoTokenizer
            print(f"🔄 加载 Reranker: {self.model_name}")
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(
                self.model_name, torch_dtype="auto")
            self.model.eval()
            self._ready = True
            print(f"✅ Reranker 加载完成")
            return True
        except Exception as e:
            print(f"⚠️ Reranker 加载失败: {e}")
            return False

    def rerank(self, query: str, chunks: List[Tuple],
               top_k: int = 20) -> List[Tuple]:
        """对候选结果重打分"""
        if not self._ready or not chunks:
            return chunks
        import torch
        candidates = chunks[:top_k]
        pairs = [[query, c[0].text] for c in candidates]
        inputs = self.tokenizer(pairs, padding=True, truncation=True,
                                return_tensors="pt", max_length=512)
        with torch.no_grad():
            scores = self.model(**inputs).logits.squeeze(-1).tolist()
        if isinstance(scores, float):
            scores = [scores]
        scored = [(ch, float(s), src) for (ch, _, src), s in zip(candidates, scores)]
        scored.sort(key=lambda x: -x[1])
        return scored


# 演示（首次运行会下载模型，耗时较长）
reranker = Reranker()
loaded = reranker.load()
print(f"\nReranker 状态: {'✅ 可用' if loaded else '⚠️ 未加载'}")


## 10. 生成层 — DeepSeek LLM 集成

**流程**：检索到的片段 + 元数据 → 构造 Prompt → DeepSeek API → 结构化答案

**Prompt 工程要点**：
- temperature=0.1：业务场景需确定性，不宜高
- 要求引用原文并注明页码
- 涉及变更时必须输出对比表格
- 信息不足时必须如实说明

In [ ]:
from openai import OpenAI

# 从环境变量或 .env 读取 API Key
import os
if os.path.exists(".env"):
    for line in open(".env"):
        if "=" in line:
            k, v = line.strip().split("=", 1)
            os.environ[k] = v

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "sk-your-key-here")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_MODEL = "deepseek-chat"

client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)

# ── Prompt 模板 ──

PROMPT_TEMPLATE = """你是一个保险业务专家，专门回答保单受益人变更相关的问题。

## 检索到的相关片段

{context}

## 元数据信息

{metadata}

## 用户问题

{question}

## 回答要求
1. 仅基于上述检索片段回答，如果信息不足请如实说明
2. 引用原文片段和页码，按 JSON 格式组织答案
3. 涉及变更时必须输出变更前/后的对比表格
4. 使用专业、清晰的中文
"""


def build_context(chunks: List[Tuple]) -> str:
    """将检索结果组装为上下文文本"""
    lines = []
    for i, (ch, score, src) in enumerate(chunks, 1):
        lines.append(
            f"[{i}] (保单{ch.pid}, 第{ch.page}页, 段落:{ch.type}, "
            f"heading:{ch.heading})\n{ch.text}"
        )
    return "\n\n".join(lines)


def build_metadata(chunks: List[Tuple]) -> str:
    """提取相关保单的元数据信息"""
    seen = set()
    meta_lines = []
    for ch, _, _ in chunks:
        pid = ch.pid
        if pid not in seen and pid in all_metas:
            seen.add(pid)
            meta = all_metas[pid]
            meta_lines.append(str(meta.to_dict()))
    return "\n".join(meta_lines)


def answer_question(question: str, chunks: List[Tuple]) -> str:
    """调用 DeepSeek 生成答案"""
    context = build_context(chunks)
    metadata = build_metadata(chunks)
    prompt = PROMPT_TEMPLATE.format(
        context=context, metadata=metadata, question=question
    )
    resp = client.chat.completions.create(
        model=DEEPSEEK_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=2000,
    )
    return resp.choices[0].message.content


# ── 测试问答 ──
print("🧪 测试问答：'张美玲的受益比例是多少？'\n")

# 1) 混合检索
results = hybrid_search("张美玲受益比例", policy_id="P0242025-1883")
print(f"检索到 {len(results)} 个相关片段\n")

# 2) 生成答案
if DEEPSEEK_API_KEY and DEEPSEEK_API_KEY != "sk-your-key-here":
    answer = answer_question("张美玲的受益比例是多少？", results)
    print("=" * 60)
    print(answer)
else:
    print("⚠️ 未配置 DEEPSEEK_API_KEY，跳过 API 调用")
    print("请在 .env 文件中设置 DEEPSEEK_API_KEY=sk-xxxx")


## 11. 服务层 — FastAPI

**架构图**：
```
Client → FastAPI → RetrievalEngine → PGVector / Memory
                  → Reranker
                  → DeepSeek API → Response
```

将上述组件封装为 Web 服务，提供 RESTful API。


In [ ]:
import json
from fastapi import FastAPI, UploadFile, File, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# ========== Pydantic 请求/响应模型 ==========

class SearchRequest(BaseModel):
    query: str
    policy_id: Optional[str] = None
    top_k: int = 5

class SearchResponse(BaseModel):
    query: str
    results: List[dict]
    total: int

class AnswerRequest(BaseModel):
    query: str
    policy_id: Optional[str] = None

class AnswerResponse(BaseModel):
    query: str
    answer: str
    references: List[dict]

class HealthResponse(BaseModel):
    status: str
    vector_db: str
    fulltext_db: str
    reranker: str
    total_policies: int
    total_chunks: int

# ========== FastAPI 应用 ==========

app = FastAPI(title="保单受益人变更检索系统", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# 全局状态
search_engine = {"ready": False}

@app.on_event("startup")
async def startup():
    # 检索引擎初始化（加载模型 / 连接数据库）
    search_engine["embedder"] = embedder
    search_engine["bm25"] = bm25
    search_engine["chunks"] = children
    search_engine["all_chunks"] = all_chunks
    search_engine["all_metas"] = all_metas
    search_engine["dense_matrix"] = dense_matrix
    search_engine["ready"] = True
    print("✅ FastAPI 就绪")

# ========== REST API 端点 ==========

@app.get("/api/v1/health", response_model=HealthResponse)
async def health():
    return HealthResponse(
        status="ok",
        vector_db="memory (numpy)",
        fulltext_db="memory (rank_bm25)",
        reranker="active" if reranker._ready else "inactive",
        total_policies=len(all_metas),
        total_chunks=len(children),
    )

@app.post("/api/v1/search", response_model=SearchResponse)
async def search(req: SearchRequest):
    raw = hybrid_search(req.query, req.policy_id, req.top_k)
    results = []
    for ch, score, src in raw:
        meta = all_metas.get(ch.pid)
        results.append({
            "chunk_id": ch.cid,
            "policy_id": ch.pid,
            "page": ch.page,
            "type": ch.type,
            "heading": ch.heading,
            "text": ch.text,
            "score": round(score, 4),
            "source": src,
            "metadata": meta.to_dict() if meta else None,
        })
    return SearchResponse(query=req.query, results=results, total=len(results))

@app.post("/api/v1/answer", response_model=AnswerResponse)
async def answer(req: AnswerRequest):
    raw = hybrid_search(req.query, req.policy_id, 5)
    if reranker._ready:
        raw = reranker.rerank(req.query, raw, 5)

    refs = []
    for ch, score, src in raw:
        meta = all_metas.get(ch.pid)
        refs.append({
            "policy_id": ch.pid, "page": ch.page,
            "heading": ch.heading, "text": ch.text,
            "score": round(score, 4),
        })

    # 生成答案
    if DEEPSEEK_API_KEY and DEEPSEEK_API_KEY != "sk-your-key-here":
        ans_text = answer_question(req.query, raw)
    else:
        ans_text = "⚠️ 未配置 DEEPSEEK_API_KEY，跳过 LLM 生成。请设置环境变量后重试。"

    return AnswerResponse(query=req.query, answer=ans_text, references=refs)

print("✅ FastAPI 应用定义完成")
print(f"   路由: /api/v1/health, /api/v1/search, /api/v1/answer")
print(f"   API Key 配置: {'✅ 已设置' if DEEPSEEK_API_KEY and DEEPSEEK_API_KEY != 'sk-your-key-here' else '⚠️ 未设置'}")


## 12. 前端层 — Streamlit UI

**功能**：搜索输入 → 调用 FastAPI → 展示结果

启动方式：
```bash
cd rag_service
pip install streamlit requests
streamlit run frontend.py --server.port 8501
```


In [ ]:
# ── Streamlit 前端代码（参考实现）──
# 完整文件见 rag_service/frontend.py
# 此处仅展示核心逻辑

STREAMLIT_CODE = '''
import streamlit as st
import requests

API_BASE = "http://localhost:8000/api/v1"

st.set_page_config(page_title="保单受益人变更检索系统", layout="wide")
st.title("📄 保单受益人变更检索系统")

query = st.text_input("🔍 输入查询内容", placeholder="例如：张美玲受益比例")
col1, col2 = st.columns([1, 3])

with col1:
    if st.button("搜索", type="primary") and query:
        # 调用检索
        resp = requests.post(f"{API_BASE}/search",
            json={"query": query, "policy_id": "", "top_k": 5}).json()
        st.subheader(f"📎 检索结果 ({resp['total']})")
        for r in resp["results"]:
            with st.expander(f"保单{r['policy_id']} 第{r['page']}页 [{r['heading']}]"):
                st.caption(f"得分: {r['score']}")
                st.markdown(r["text"])

with col2:
    if st.button("问答", type="secondary") and query:
        resp = requests.post(f"{API_BASE}/answer",
            json={"query": query}).json()
        st.subheader("💡 答案")
        st.markdown(resp["answer"])
'''

print("✅ Streamlit 前端逻辑展示如上")
print("   （请参考 rag_service/frontend.py 获取完整实现）")


## 13. 生产级存储 — PostgreSQL + pgvector

**为什么从内存迁移到 PG？**

| 组件 | 内存版本（本教程） | 生产版本 |
|------|-------------------|---------|
| 向量存储 | numpy 矩阵 | pgvector (cosine) |
| 全文检索 | rank_bm25 | jieba + GIN index |
| 持久化 | ❌ 重启丢失 | ✅ 持久存储 |
| 并发 | ❌ 单进程 | ✅ 多连接 |

**向量索引**：IVFFlat（Inverted File with Flat）— 将向量空间划分为单元，搜索时只扫描最近几个单元的向量。


In [ ]:
# ── PostgreSQL pgvector 集成示例 ──

PG_CONFIG = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", 5432)),
    "dbname": os.getenv("PGDATABASE", "rag_poc"),
    "user": os.getenv("PGUSER", "rag_user"),
    "password": os.getenv("PGPASSWORD", "rag_pass"),
}


def pg_connect():
    """连接 PostgreSQL"""
    try:
        import psycopg2
        conn = psycopg2.connect(**PG_CONFIG)
        conn.autocommit = True
        return conn
    except Exception as e:
        print(f"⚠️ PostgreSQL 连接失败: {e}")
        return None


# 尝试连接（本地可能没有 PG，预告概念即可）
pg_conn = pg_connect()
if pg_conn:
    cur = pg_conn.cursor()
    # 1) 启用 pgvector 扩展
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
    print("✅ vector 扩展已启用")

    # 2) 创建表结构
    cur.execute("""
        CREATE TABLE IF NOT EXISTS chunks (
            id SERIAL PRIMARY KEY,
            chunk_id VARCHAR(64) UNIQUE,
            policy_id VARCHAR(64),
            page INT,
            chunk_type VARCHAR(32),
            heading TEXT,
            text TEXT NOT NULL,
            embedding vector(384),
            tokens TEXT[],
            metadata JSONB DEFAULT '{}'
        )
    """)
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_chunks_policy
        ON chunks(policy_id)
    """)
    # 3) IVFFlat 索引（需有数据后才能创建）
    cur.execute("""
        SELECT 1 FROM chunks LIMIT 1
    """)
    has_data = cur.fetchone() is not None
    if has_data:
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_chunks_embedding
            ON chunks USING ivfflat (embedding vector_cosine_ops)
            WITH (lists = 100)
        """)
        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_chunks_tokens
            ON chunks USING gin (tokens)
        """)
        print("✅ 向量索引 + GIN 索引已创建")
    else:
        print("ℹ️ 表已创建，暂无数据（数据插入后再建索引）")

    # 4) 插入数据演示
    cur.execute("SELECT COUNT(*) FROM chunks")
    count = cur.fetchone()[0]
    if count == 0 and dense_matrix is not None:
        print(f"🔄 插入 {len(children)} 条数据到 PG...")
        for i, ch in enumerate(children):
            vec = dense_matrix[i].tolist()
            tokens = tokenize(ch.text)
            cur.execute(
                """INSERT INTO chunks
                   (chunk_id, policy_id, page, chunk_type, heading, text, embedding, tokens)
                   VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
                   ON CONFLICT DO NOTHING""",
                (ch.cid, ch.pid, ch.page, ch.type, ch.heading, ch.text, vec, tokens)
            )
        print(f"✅ 插入完成")

    # 5) 向量搜索演示
    q = "张美玲受益比例"
    qv = embedder.encode([q])[0].tolist()
    cur.execute(
        """SELECT chunk_id, policy_id, page, text,
                   (embedding <=> %s::vector) AS distance
           FROM chunks
           ORDER BY distance
           LIMIT 3""",
        (qv,)
    )
    pg_results = cur.fetchall()
    print(f"\n🔍 PG 向量搜索 '{q}' 结果：")
    for r in pg_results:
        print(f"   [{r[1]}] 第{r[2]}页 | 距离={r[4]:.4f} | {r[3][:60]}...")

    # 6) 全文搜索演示
    qt = tokenize(q)
    cur.execute(
        """SELECT chunk_id, policy_id, page, text
           FROM chunks
           WHERE tokens && %s
           LIMIT 3""",
        (qt,)
    )
    ft_results = cur.fetchall()
    print(f"\n🔍 PG 全文搜索 '{q}' 结果：")
    for r in ft_results:
        print(f"   [{r[1]}] 第{r[2]}页 | {r[3][:60]}...")

    cur.close()
else:
    print("\n⚠️ 未连接到 PostgreSQL，跳过 PG 集成演示。")
    print("   启动方式：")
    print("   docker run -d --name pg-rag \\")
    print("     -e POSTGRES_DB=rag_poc -e POSTGRES_USER=rag_user \\")
    print("     -e POSTGRES_PASSWORD=rag_pass \\")
    print("     -p 5432:5432 pgvector/pgvector:pg16")
    print("\n   或参考 docker-compose.yml 使用 Docker Compose 启动")


## 14. 部署 — Docker Compose

**服务架构**：
```
┌──────────┐     ┌──────────┐     ┌──────────┐
│ Frontend │────▶│   API    │────▶│ Postgres │
│ :8501    │     │ :8000    │     │ :5432    │
└──────────┘     └──────────┘     └──────────┘
```

使用 `docker-compose up -d` 即可一键启动全部服务。


In [ ]:
# ── Docker Compose 配置（参考）──

DOCKER_COMPOSE_YML = """
version: "3.8"

services:
  postgres:
    image: pgvector/pgvector:pg16
    environment:
      POSTGRES_DB: rag_poc
      POSTGRES_USER: rag_user
      POSTGRES_PASSWORD: rag_pass
    ports:
      - "5432:5432"
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U rag_user -d rag_poc"]
      interval: 5s
      timeout: 5s
      retries: 5

  api:
    build: .
    ports:
      - "8000:8000"
    env_file: .env
    environment:
      PGHOST: postgres
    depends_on:
      postgres:
        condition: service_healthy
    volumes:
      - ./data:/app/data  # 模型缓存

  frontend:
    build:
      context: .
      dockerfile: Dockerfile.frontend
    ports:
      - "8501:8501"
    environment:
      API_BASE: http://api:8000
    depends_on:
      - api
"""

print("✅ Docker Compose 配置参考")
print("   启动: cd rag_service && docker compose up -d")
print("   停止: docker compose down")
print("   查看日志: docker compose logs -f")


## 15. 端到端验证

**验证清单**：

| # | 测试项 | 预期结果 |
|---|--------|---------|
| 1 | Health 检查 | 返回系统状态，各组件正常 |
| 2 | 混合检索 | 返回 RRF 融合排序结果 |
| 3 | Reranker 精排 | Cross-encoder 重打分 |
| 4 | LLM 问答 | DeepSeek 返回结构化答案 |
| 5 | 元数据提取 | 正确识别保单号/受益人等信息 |
| 6 | Docker 部署 | 3 个容器全部 healthy |


In [ ]:
# ── 端到端验证函数 ──

def run_e2e_tests():
    """运行完整的端到端测试"""
    passed = 0
    total = 0

    def check(name, condition, detail=""):
        nonlocal passed, total
        total += 1
        status = "✅" if condition else "❌"
        if condition:
            passed += 1
        print(f"  {status} {name}  {detail}")

    print("=" * 60)
    print("🧪 端到端验证")
    print("=" * 60)

    # Test 1: 元数据提取
    print("\n📋 1. 元数据提取测试")
    meta_a = extract_metadata(POLICIES[0])
    check("保单号提取", meta_a.policy_id == "P0242025-1883",
          f"got={meta_a.policy_id}")
    check("受益人识别", len(meta_a.beneficiaries) > 0,
          f"count={len(meta_a.beneficiaries)}")

    # Test 2: 分块
    print("\n📋 2. 分块测试")
    chunks_a = chunk_policy(POLICIES[0])
    children_a = [c for c in chunks_a if c.type == "child"]
    check("子块数量>0", len(children_a) > 0, f"count={len(children_a)}")
    check("父块存在", len(chunks_a) > len(children_a),
          f"total={len(chunks_a)}, children={len(children_a)}")

    # Test 3: Embedding
    print("\n📋 3. Embedding 测试")
    check("向量维度=384", dense_matrix.shape[1] == 384,
          f"got={dense_matrix.shape[1]}")
    check("向量数量=子块数", dense_matrix.shape[0] == len(children),
          f"got={dense_matrix.shape[0]}")

    # Test 4: 混合检索
    print("\n📋 4. 混合检索测试")
    res = hybrid_search("张美玲受益比例")
    check("返回结果>0", len(res) > 0, f"count={len(res)}")
    check("包含得分", all(r[1] > 0 for r in res), "all scores > 0")
    check("按得分降序", all(res[i][1] >= res[i+1][1] for i in range(len(res)-1)),
          "scores descending")

    # Test 5: BM25 + 向量各自工作
    print("\n📋 5. 检索组件测试")
    dense_res = _dense_search("张美玲")
    bm25_res = _bm25_search("张美玲")
    check("向量检索有效", len(dense_res) > 0, f"count={len(dense_res)}")
    check("BM25检索有效", len(bm25_res) > 0, f"count={len(bm25_res)}")

    # Test 6: 策略过滤
    print("\n📋 6. 保单过滤测试")
    filtered = hybrid_search("变更", policy_id="P0242025-1883")
    check("仅返回指定保单", all(r[0].pid == "P0242025-1883" for r in filtered),
          f"all pid=P0242025-1883")

    # Test 7: LLM 集成
    print("\n📋 7. LLM 集成测试")
    if DEEPSEEK_API_KEY and DEEPSEEK_API_KEY != "sk-your-key-here":
        ans = answer_question("张美玲受益比例是多少？", res)
        check("LLM 返回非空", len(ans) > 0, f"len={len(ans)}")
    else:
        print("  ⏭️ 跳过 LLM 测试（未配置 API Key）")

    # 汇总
    print(f"\n{'=' * 60}")
    print(f"📊 结果: {passed}/{total} 通过"
          + (" 🎉" if passed == total else " ⚠️ 有失败"))
    print(f"{'=' * 60}")
    return passed == total


# 运行测试
all_ok = run_e2e_tests()
print(f"\n全量测试结果: {'🎉 全部通过!' if all_ok else '⚠️ 请检查失败项'}")


## 总结

### 完整架构回顾

```
┌─────────────────────────────────────────────────────┐
│                   生成层 (Generation)                 │
│         DeepSeek API + Prompt Template               │
│         结构化答案 + 变更对比表格                      │
├─────────────────────────────────────────────────────┤
│                   检索层 (Retrieval)                   │
│   RRF(α·Dense + (1-α)·BM25) → Reranker → Top-5      │
├─────────────────────────────────────────────────────┤
│                   索引层 (Index)                       │
│   Embedding(384-dim) + BM25(jieba) + pgvector        │
├─────────────────────────────────────────────────────┤
│                   ETL层 (Extract)                     │
│   元数据提取(正则) → Parent-Child分块 → 结构化存储    │
└─────────────────────────────────────────────────────┘
```

### 核心公式

**RRF 融合得分**：
$$Score(chunk) = \alpha \cdot \frac{1}{rank_{dense} + K} + (1-\alpha) \cdot \frac{1}{rank_{bm25} + K}$$

**余弦相似度**：
$$sim(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

### 文件结构

| 文件 | 用途 |
|------|------|
| `config.py` | 全局配置（API Key、模型名、检索参数） |
| `models.py` | 数据模型（PolicyMeta、Chunk、请求/响应） |
| `data.py` | 模拟保单数据（3 份示例） |
| `engine.py` | 检索引擎（Embedding、BM25、RRF、Reranker） |
| `reranker.py` | Cross-encoder 精排封装 |
| `llm.py` | DeepSeek API 集成 |
| `app.py` | FastAPI 服务（REST API） |
| `frontend.py` | Streamlit 前端 UI |
| `docker-compose.yml` | 三容器编排（PG + API + Frontend） |

---

**🎉 恭喜！你已完整掌握了保险 RAG 系统的原理与实现！**
